In [14]:
%%capture
!pip install transformers

In [15]:
import torch
import torch.nn.functional as F
from transformers import ElectraTokenizer, ElectraModel
import requests
import pandas as pd
import numpy as np
import random
import string

stop_wrds = ["!","\"","#",".","$","%","&","\'","(",")","*","+",",", "-",".","/",":",";","<","=",">","?","@","[","\\","\]","^","_","`","{","|","}","~","[CLS]","[PAD]","[SEP]","।"]

class KeywordExtractor:
    def __init__(self):
        self.tokenizer = None
        self.model = None

    def load_model(self):
        model_name = 'csebuetnlp/banglabert'
        self.tokenizer = ElectraTokenizer.from_pretrained(model_name)
        self.model = ElectraModel.from_pretrained(model_name)
        self.model.eval()

    def score_words(self, sentence):
        input_ids = torch.tensor([self.tokenizer.encode(sentence, max_length=512, padding='max_length', add_special_tokens=True, truncation=True)])
        tokenized_text = self.tokenizer.convert_ids_to_tokens(input_ids[0])

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids)

        embeddings = outputs.last_hidden_state.squeeze(0)
        mean_embedding = embeddings.mean(dim=0)

        original_words = []
        original_word_embeddings = []
        current_word = ""
        prev_word = ""
        prev_embedding = ""
        current_embedding_lst = []

        for i, embedding in enumerate(embeddings):

            token = tokenized_text[i]
            if token in stop_wrds:
              continue
            elif token.startswith("##"):
                current_word = prev_word + token[2:]
                current_embedding_lst.append(prev_embedding)
                current_embedding_lst.append(embedding)
                current_embedding = sum(current_embedding_lst)/len(current_embedding_lst)

                original_words.pop()
                original_word_embeddings.pop()

                original_words.append(current_word)
                original_word_embeddings.append(current_embedding)

                prev_word = current_word
                prev_embedding = current_embedding
                current_embedding_lst = []
            else:
                original_words.append(token)
                original_word_embeddings.append(embedding)
                prev_word = token
                prev_embedding = embedding

        # for i, (token, embedding) in enumerate(zip(original_words, original_word_embeddings)):
        #   print(token)
        #   print(embedding.shape)

        word_scores = []
        new_scores = []
        for i in range(len(original_word_embeddings)):
            cos_sim = torch.nn.functional.cosine_similarity(original_word_embeddings[i], mean_embedding, dim=0)
            word_scores.append((original_words[i], cos_sim.item()))
            new_scores.append((original_words[i], original_word_embeddings[i], cos_sim.item()))

        return word_scores, mean_embedding, new_scores

    def print_top_values(self, data):
        top_values = int(len(data) * 0.6)
        if top_values < 10:
            top_values = int(len(data) * 0.7)
        if top_values < 4:
            top_values = int(len(data) * 0.8)

        finalLst = []
        for i in range(top_values):
            finalLst.append(data[i][0])

        return finalLst

    def keysOfSentence(self, sentence):
        final = []
        word_scores, mean_embedding, new_scores = self.score_words(sentence)
        word_scores.sort(key = lambda x: x[1], reverse=True)
        new_scores.sort(key = lambda x: x[2], reverse=True)

        finalKeysLst = self.print_top_values(word_scores)
        finalKeys = finalKeysLst

        return finalKeys, mean_embedding, new_scores

    def extract_keywords(self, text):
        self.load_model()
        return self.keysOfSentence(text)

extractor = KeywordExtractor()

In [25]:
# Example usage
text = "শূন্য আসনে গত ২৯ মার্চ উপনির্বাচন অনুষ্ঠিত হয়।"
keywords, mean_embedding, new_scores = extractor.extract_keywords(text)
print(keywords)

['অনুষ্ঠিত', 'আসনে', 'উপনির্বাচন', 'হয়', 'শূন্য']


In [26]:
for i in range(len(new_scores)):
    print(new_scores[i][0])
    print(new_scores[i][2])

অনুষ্ঠিত
0.645639181137085
আসনে
0.5570165514945984
উপনির্বাচন
0.5198491215705872
হয়
0.45861899852752686
শূন্য
0.45375359058380127
মার্চ
0.4164905548095703
গত
0.38293248414993286
২৯
0.3766247034072876


In [18]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rc
from collections import defaultdict
from textwrap import wrap
from pylab import rcParams

from matplotlib.font_manager import fontManager, FontProperties

# -*- coding: utf-8 -*-
from __future__ import unicode_literals

In [19]:
%%capture
!pip install umap-learn
!pip install plotly

In [20]:
import umap
from umap import UMAP
import plotly.express as px

def TSNE_Func(mean_embedding, word_embedding, perplex_score):

  embedding_list = []
  embedding_list.append(mean_embedding.tolist())
  label_list = []
  label_list.append("mean")

  print(len(word_embedding))

  for i in range(len(word_embedding)):
     embedding_list.append(word_embedding[i][1].tolist())
     label_list.append(word_embedding[i][0])

  print(len(embedding_list))

  standardized_data = StandardScaler().fit_transform(np.asarray(embedding_list))
  tsne_model = TSNE(n_components = 2, perplexity = perplex_score, random_state = 0)
  tsne_data = tsne_model.fit_transform(np.asarray(standardized_data))
  # umap_3d = UMAP(n_components=3, init='random', random_state=42)
  # proj_3d = umap_3d.fit_transform(embedding_list)

  # # Plotting the result of tsne
  # fig_3d = px.scatter_3d(
  #     proj_3d, x=0, y=1, z=2,
  #     color=label_list, labels={'color': 'embedding'}
  # )
  # fig_3d.update_traces(marker_size=5)

  # fig_3d.show()

  # umap_2d = UMAP(n_components=2, init='random', random_state=42)
  # proj_2d = umap_2d.fit_transform(embedding_list)

  # Plotting the result of tsne
  fig_2d = px.scatter(
      tsne_data, x=0, y=1,
      color=label_list, labels={'color': 'embedding'}
  )

  fig_2d.show()

In [28]:
word_embedd = []
for i in range(len(new_scores)):
  if new_scores[i][0] in keywords:
    word_embedd.append(new_scores[i])

TSNE_Func(mean_embedding, word_embedd, len(keywords)-1)

5
6
